# Topic 02 — GPU profiling and inference bottlenecks

This lab separates **load**, **warm-up**, **prefill**, and **decode**. It records exact model revisions, token shapes, synchronized CUDA timings, memory counters, KV-cache arithmetic, and a profiler trace. The goal is a defensible diagnosis, not the fastest number.

> Runtime: a CUDA GPU with at least 16 GiB is recommended. Run one model at a time. Restart the runtime if fragmentation prevents the second model from loading.

In [ ]:
%pip install -q "transformers==4.51.3" "accelerate==1.6.0" "safetensors==0.5.3"


In [ ]:
import gc, json, os, platform, statistics, time
from pathlib import Path
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'A CUDA runtime is required for the measured lab'
DEVICE = torch.device('cuda')
DTYPE = torch.float16
TRIALS = 5
MODELS = [
    {'id': 'Qwen/Qwen2.5-0.5B-Instruct', 'revision': '7ae557604adf67be50417f59c2c2f167def9a775'},
    {'id': 'Qwen/Qwen2.5-1.5B-Instruct', 'revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'},
]
WORKLOADS = [
    {'prompt_tokens': 128, 'output_tokens': 32},
    {'prompt_tokens': 4096, 'output_tokens': 32},
    {'prompt_tokens': 128, 'output_tokens': 128},
    {'prompt_tokens': 4096, 'output_tokens': 128},
]
RUN = {
    'metric_contract': 1, 'utc_started': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__,
    'cuda_runtime': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
    'trials': TRIALS, 'dtype': str(DTYPE), 'models': MODELS, 'workloads': WORKLOADS, 'results': []
}
RUN


## Measurement helpers
CUDA work is asynchronous, so the timer synchronizes before and after each region. Inputs are constructed as token IDs; no decode/re-tokenize estimate is used. Prefill and decode are measured separately.

In [ ]:
def timed_cuda(fn):
    torch.cuda.synchronize()
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record(); value = fn(); end.record()
    torch.cuda.synchronize()
    return value, start.elapsed_time(end)

def exact_input(tokenizer, length):
    seed = tokenizer.encode(' inference systems need measured evidence', add_special_tokens=False)
    ids = (seed * ((length + len(seed) - 1) // len(seed)))[:length]
    return torch.tensor([ids], device=DEVICE, dtype=torch.long)

def kv_bytes_per_token(config, dtype):
    layers = config.num_hidden_layers
    kv_heads = getattr(config, 'num_key_value_heads', config.num_attention_heads)
    head_dim = getattr(config, 'head_dim', config.hidden_size // config.num_attention_heads)
    element_bytes = torch.tensor([], dtype=dtype).element_size()
    return {'layers': layers, 'kv_heads': kv_heads, 'head_dim': head_dim,
            'element_bytes': element_bytes,
            'bytes_per_token': 2 * layers * kv_heads * head_dim * element_bytes}

@torch.inference_mode()
def measure(model, tokenizer, workload):
    ids = exact_input(tokenizer, workload['prompt_tokens'])
    warmup = model(ids[:, :min(16, ids.shape[1])], use_cache=True)  # unmeasured warm-up
    del warmup
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    prefill_ms, decode_ms = [], []
    for _ in range(TRIALS):
        out, ms = timed_cuda(lambda: model(ids, use_cache=True))
        prefill_ms.append(ms)
        past, next_id = out.past_key_values, out.logits[:, -1:].argmax(-1)
        def decode_loop():
            nonlocal past, next_id
            for _ in range(workload['output_tokens'] - 1):  # prefill logits select token 1
                step = model(next_id, past_key_values=past, use_cache=True)
                past, next_id = step.past_key_values, step.logits[:, -1:].argmax(-1)
        _, ms = timed_cuda(decode_loop); decode_ms.append(ms)
        del out, past; gc.collect(); torch.cuda.empty_cache()
    return {
        **workload, 'batch_size': 1, 'trials': TRIALS,
        'prefill_ms': prefill_ms, 'prefill_median_ms': statistics.median(prefill_ms),
        'decode_steps': workload['output_tokens'] - 1,
        'decode_ms': decode_ms, 'decode_median_ms': statistics.median(decode_ms),
        'decode_tokens_per_s': (workload['output_tokens'] - 1) / (statistics.median(decode_ms) / 1000),
        'memory_allocated_bytes': torch.cuda.memory_allocated(),
        'memory_reserved_bytes': torch.cuda.memory_reserved(),
        'peak_allocated_bytes': torch.cuda.max_memory_allocated(),
        'peak_reserved_bytes': torch.cuda.max_memory_reserved(),
    }


## Run the matrix
Each model is unloaded before the next is loaded. `revision` is a full immutable commit. The JSON evidence stores every trial, not only the median.

In [ ]:
for spec in MODELS:
    torch.cuda.empty_cache(); gc.collect()
    tokenizer = AutoTokenizer.from_pretrained(spec['id'], revision=spec['revision'])
    load_started = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(
        spec['id'], revision=spec['revision'], torch_dtype=DTYPE, low_cpu_mem_usage=True
    ).to(DEVICE).eval()
    load_s = time.perf_counter() - load_started
    record = {'model': spec, 'load_s': load_s, 'kv': kv_bytes_per_token(model.config, DTYPE), 'measurements': []}
    for workload in WORKLOADS:
        result = measure(model, tokenizer, workload)
        record['measurements'].append(result); print(spec['id'], result)
    RUN['results'].append(record)
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()

Path('evidence').mkdir(exist_ok=True)
Path('evidence/topic-02-benchmark.json').write_text(json.dumps(RUN, indent=2), encoding='utf-8')
print('wrote evidence/topic-02-benchmark.json')


## Focused profiler trace
A profiler trace answers *where the measured region spent time*; profiler overhead means it is not the production latency measurement. The trace below is deliberately short.

In [ ]:
spec = MODELS[0]
tokenizer = AutoTokenizer.from_pretrained(spec['id'], revision=spec['revision'])
model = AutoModelForCausalLM.from_pretrained(spec['id'], revision=spec['revision'], torch_dtype=DTYPE).to(DEVICE).eval()
ids = exact_input(tokenizer, 128)
with torch.inference_mode(), torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True, profile_memory=True
) as prof:
    with torch.profiler.record_function('topic02_prefill_128'):
        _ = model(ids, use_cache=True)
    torch.cuda.synchronize()
prof.export_chrome_trace('evidence/topic-02-trace.json')
print(prof.key_averages().table(sort_by='self_cuda_time_total', row_limit=15))


## Analysis contract
Create `evidence/topic-02-analysis.md`. For both models: reconcile the config-derived KV bytes/token with the workload; compare short/long prefill and decode; name the likely limiter and one alternative hypothesis; cite trace evidence; state omissions (kernel fusion, allocator state, clocking, thermal state, host overhead, and batch size); propose the next discriminating experiment. Never infer compute saturation from GPU-utilization percentage alone.